<a href="https://colab.research.google.com/github/Aswanth0704/gpu-programming-cpp/blob/main/6_MemorySpaces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

if os.getenv("COLAB_RELEASE_TAG"): # If running in Google Colab:
  !mkdir -p Sources
  !wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.06-Memory-Spaces/Sources/ach.h -nv -O Sources/ach.h

2026-09-26 20:09:43 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.06-Memory-Spaces/Sources/ach.h [3296/3296] -> "Sources/ach.h" [1]


In [6]:
%%writefile Sources/heat-2D.cpp
#include "ach.h"

int main()
{
  int height = 4096;
  int width = 4096;

  thrust::universal_vector<float> prev = ach::init(height, width);
  thrust::universal_vector<float> next(height*width);

  for (int write_step = 0; write_step < 3; write_step ++)
  {
    std::printf("  write step %d\n", write_step);
    ach::store(write_step, height, width, prev);

    for (int compute_step = 0; compute_step < 3; compute_step++)
    {
      auto begin = std::chrono::high_resolution_clock::now();
      ach::simulate(height, width, prev, next);
      auto end = std::chrono::high_resolution_clock::now();
      auto seconds = std::chrono::duration<double>(end-begin).count();
      std::printf("computed step %d in %g s\n", compute_step, seconds);
      prev.swap(next);
    }
  }
}

Overwriting Sources/heat-2D.cpp


In [7]:
!nvcc --extended-lambda -o /tmp/a.out Sources/heat-2D.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

  write step 0
computed step 0 in 0.0190118 s
computed step 1 in 0.000601124 s
computed step 2 in 0.000592765 s
  write step 1
computed step 0 in 0.0454008 s
computed step 1 in 0.000612437 s
computed step 2 in 0.00060134 s
  write step 2
computed step 0 in 0.0360605 s
computed step 1 in 0.000613496 s
computed step 2 in 0.000587482 s


performance after each store is degrading because the universal vectors which are copied to and fro from the CPU and GPU. copying from GPU to CPU for storage is fine, but we don't want to copy it back. This is done using host and device memory spaces

- We should not just think about where to run code, but also, where the bytes live.


In [ ]:
%%writefile Sources/memoryspaces.cpp
thrust::host_vector<int> h_vec{11, 12};
thrust::device_vector<int> d_vec{21, 22};
thrust::copy_n(h_vec.begin(), 1, d_vec.begin());